In [1]:
import json
import pandas as pd

import sys, os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from shared_functions.gg_sheet_drive import *

d:\miniconda3\envs\only_env\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


#### PubmedQA

In [ ]:
from datasets import load_dataset

pqaa = load_dataset("pubmed_qa", "pqa_artificial")
pqal = load_dataset("pubmed_qa", "pqa_labeled")
pqau = load_dataset("pubmed_qa", "pqa_unlabeled")

In [ ]:
pqaa = pqaa['train']
pqal = pqal['train']
pqau = pqau['train']

pqaa = pd.DataFrame(pqaa)
pqau = pd.DataFrame(pqau)
pqal = pd.DataFrame(pqal)

pqau['final_decision'] = ""

In [ ]:
df = pd.concat([pqaa, pqau, pqal], axis = 0)

df['final_decision'] = df['final_decision'].replace('', 'not_given')

In [ ]:
df.to_csv('pubmedqa.csv', index = False)

#### BioASQ

In [ ]:
bioasq_file = os.path.join(project_root, 'data', 'bioasq', 'task_b', 'training14b.json')

with open(bioasq_file, 'r', encoding='utf-8') as f:
    bioasq_data = json.load(f)

questions = bioasq_data.get('questions', [])
df_bioasq = pd.json_normalize(questions)

df = df_bioasq[['body', 'documents', 'ideal_answer', 'concepts', 'type', 'id',
       'snippets', 'triples', 'exact_answer']]

#### MedQA

In [ ]:
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf8") as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

dataset_dir = os.path.join(project_root, 'data', 'medqa', 'questions')

train_df = load_jsonl(os.path.join(dataset_dir, '4_options', "train.jsonl"))
dev_df   = load_jsonl(os.path.join(dataset_dir, '4_options', "dev.jsonl"))
test_df  = load_jsonl(os.path.join(dataset_dir, '4_options', "test.jsonl"))

train_ex_df = load_jsonl(os.path.join(dataset_dir, "train.jsonl"))
dev_ex_df   = load_jsonl(os.path.join(dataset_dir, "dev.jsonl"))
test_ex_df  = load_jsonl(os.path.join(dataset_dir, "test.jsonl"))
qbank_df = load_jsonl(os.path.join(dataset_dir, "US_qbank.jsonl"))

df = pd.concat([train_df, dev_df, test_df, train_ex_df, dev_ex_df, test_ex_df, qbank_df], axis = 0)

df["answer_idx"] = df["answer_idx"].fillna(df["answer"])

df['processed'] = 1 - (df['metamap_phrases'].isnull())

In [ ]:
df_clean = pd.DataFrame({
    "question": df["question"],
    "option_A": df["options"].apply(lambda x: x.get("A")),
    "option_B": df["options"].apply(lambda x: x.get("B")),
    "option_C": df["options"].apply(lambda x: x.get("C")),
    "option_D": df["options"].apply(lambda x: x.get("D")),
    "option_E": df["options"].apply(lambda x: x.get("E")),  
    "answer": df["answer_idx"]
})

df_clean["metamap"] = df["metamap_phrases"]

df_clean["processed"] = df["processed"]